# 16 OCR feedback loop

Use OCR-enriched text as the content source for unresolved canonical candidates, then rerun suggestions and feedback canonicalization.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POLICY_PATH = PROJECT_ROOT / "policy" / "SCH_fileserver_policy_v2_4.yaml"

from src.llm_suggestions import load_policy, SuggestionConfig, suggest_dataframe
from src.feedback_loop import FeedbackConfig, build_feedback_review, rerun_canonical_with_feedback, feedback_summary
from src.ocr_feedback_bridge import OCRBridgeConfig, build_ocr_augmented_candidates, ocr_bridge_summary

policy = load_policy(POLICY_PATH)
print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:
def latest_output(prefix: str) -> Path:
    matches = sorted(OUTPUT_DIR.glob(f"{prefix}_*.parquet"))
    if not matches:
        raise FileNotFoundError(f"No output found for prefix: {prefix}")
    return matches[-1]

CANONICAL_PATH = latest_output("canonical_candidates")
OCR_PATH = latest_output("ocr_enriched")
print("CANONICAL_PATH =", CANONICAL_PATH.name)
print("OCR_PATH =", OCR_PATH.name)


In [ ]:
canonical = pd.read_parquet(CANONICAL_PATH)
ocr = pd.read_parquet(OCR_PATH)
augmented = build_ocr_augmented_candidates(canonical, ocr, config=OCRBridgeConfig())
display(augmented[["relative_path", "filename", "ocr_status", "ocr_bridge_applied", "bridge_text_source"]].head(20))
display(pd.DataFrame([ocr_bridge_summary(augmented)]))


In [ ]:
USE_OLLAMA = False
LLM_MAX_ROWS = 50

suggestions = suggest_dataframe(augmented, policy, config=SuggestionConfig(use_ollama=USE_OLLAMA, max_rows=LLM_MAX_ROWS, prefer_content_for_description=True))
display(suggestions[["relative_path", "suggested_description", "suggested_description_source", "suggestion_confidence"]].head(20))


In [ ]:
feedback_review = build_feedback_review(augmented, suggestions, config=FeedbackConfig(require_content_derived_description=True))
display(feedback_review[["relative_path", "accepted_field_count", "has_any_accepted_suggestion"]].head(20))
rerun = rerun_canonical_with_feedback(feedback_review, policy)
display(pd.DataFrame([feedback_summary(rerun)]))
display(rerun[["relative_path", "became_canonical_ready", "canonical_relative_path_after", "accepted_fields"]].head(20))


In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
augmented.to_parquet(OUTPUT_DIR / f"ocr_augmented_candidates_{stamp}.parquet", index=False)
augmented.to_csv(OUTPUT_DIR / f"ocr_augmented_candidates_{stamp}.csv", index=False, encoding="utf-8-sig")
suggestions.to_parquet(OUTPUT_DIR / f"ocr_llm_suggestions_{stamp}.parquet", index=False)
suggestions.to_csv(OUTPUT_DIR / f"ocr_llm_suggestions_{stamp}.csv", index=False, encoding="utf-8-sig")
feedback_review.to_parquet(OUTPUT_DIR / f"ocr_feedback_review_{stamp}.parquet", index=False)
feedback_review.to_csv(OUTPUT_DIR / f"ocr_feedback_review_{stamp}.csv", index=False, encoding="utf-8-sig")
rerun.to_parquet(OUTPUT_DIR / f"ocr_canonical_feedback_rerun_{stamp}.parquet", index=False)
rerun.to_csv(OUTPUT_DIR / f"ocr_canonical_feedback_rerun_{stamp}.csv", index=False, encoding="utf-8-sig")
print("Saved OCR-augmented suggestion + feedback outputs with stamp:", stamp)
